Reannotation of PCKO and GLKO neurons, as in 02 for WT and ECKO. Exports neuronal labels of all genotypes for the joint DEA in 07.

In [ ]:
from pathlib import Path
import pandas as pd
import scanpy as sc

In [ ]:
method_path = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/analysis/foxf2/Cellpose_1_nuclei_model")
adata = sc.read_h5ad(method_path / "adatas" / "20251110_temp-after-neuronal-subclustering.h5ad.gz")
adata = adata[adata.obs["genotype"].isin(["PCKO", "GLKO"])].copy()
sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=20, n_pcs=50)

In [ ]:
sc.tl.leiden(adata, key_added="leiden", resolution=0.25, flavor="igraph", n_iterations=2)
sc.pl.embedding(adata, basis="X_umap_harmony_20_50", color=["leiden", "cell_type"], size=250000 / adata.n_obs,
                legend_loc="on data", legend_fontweight="normal", legend_fontoutline=1, legend_fontsize=8, frameon=False)

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, standard_scale="var", n_genes=4)
sc.pl.dotplot(adata, ["Rbfox3", "Slc17a7", "Gad1", "Gad2", "Slc32a1", "Mbp", "Gfap", "Aqp4", "Cx3cr1", "Cldn5"], "leiden", standard_scale="var")

Clusters 0/12 (CA), 11 (thalamus) and 13 (DG) are outside cortex. Label them as in 02, where DG became Glut and thalamus Other:

In [ ]:
pd.crosstab(adata.obs["leiden"], adata.obs["brain_region"])

In [ ]:
ref = sc.read_h5ad(method_path / "adatas" / "20251111_temp-after-neuronal-subclustering-reannotated.h5ad.gz", backed="r").obs
g = ref[ref["cell_type"] == "Neurons-Granule-Immature"]
pd.crosstab(g["brain_region"], g["cell_type_sub"])

In [ ]:
old_to_new = {
    '0': 'Neurons-Glut',  # CA
    '1': 'Neurons-Glut',
    '2': 'Neurons-Mixed',  # oligodendrocyte signature
    '3': 'Neurons-Gaba',
    '4': 'Neurons-Gaba',
    '5': 'Neurons-Gaba',
    '6': 'Neurons-Gaba',
    '7': 'Neurons-Glut',
    '8': 'Neurons-Glut',
    '9': 'Neurons-Mixed',  # EC signature
    '10': 'Neurons-Gaba',
    '11': 'Neurons-Other',  # thalamus
    '12': 'Neurons-Glut',  # CA
    '13': 'Neurons-Glut',  # DG
    '14': 'Neurons-Mixed',  # astrocyte signature
}
adata.obs["cell_type_sub"] = adata.obs["leiden"].map(old_to_new).astype("category")
sc.pl.embedding(adata, basis="X_umap_harmony_20_50", color="cell_type_sub", size=250000 / adata.n_obs, frameon=False)

Changes in cortex compared to the original annotation:

In [ ]:
ctx = adata.obs["brain_region"] == "CTX"
pd.crosstab(adata.obs.loc[ctx, "cell_type"], adata.obs.loc[ctx, "cell_type_sub"])

In [ ]:
labels = pd.concat([ref["cell_type_sub"].astype(str), adata.obs["cell_type_sub"].astype(str)]).to_frame()
labels.to_parquet(method_path / "neuronal-subclustering" / "cell_type_sub_all_genotypes.parquet")